# Das feinjustierte Modell wird verwendet um die Datengrundlage zu klassifizieren

In [ ]:
import wandb
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, precision_score, recall_score
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification, EarlyStoppingCallback
import os
from google.colab import drive

drive.mount('/content/drive')
os.chdir("/content/drive/MyDrive/Daten")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
df = pd.read_csv("kandis_cleaned.csv")
del df["Unnamed: 0"]
del df["Unnamed: 0.1"]
df.shape[0] #Anzahl der insgesamten Fälle

In [ ]:
MODEL = "deepset/gbert-large"
model_best =  AutoModelForSequenceClassification.from_pretrained("best_model_multi")
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [ ]:
def predict_with_model(model, tokenizer, texts, device, batch_size=16):
    """
    Use the fine-tuned model to predict probabilities for the given texts
    in batches to avoid OOM errors.
    """
    model.to(device)
    model.eval()  # Ensure the model is in evaluation mode

    all_predictions = []
    all_probabilities = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        batch_texts = [str(text) for text in batch_texts]

        # Tokenize the input texts for the current batch
        encodings = tokenizer(
            batch_texts, truncation=True, padding=True, max_length=512, return_tensors="pt"
        )
        encodings = {key: tensor.to(device) for key, tensor in encodings.items()}  # Move tensors to device

        # Get predictions for the current batch
        with torch.no_grad():
            outputs = model(**encodings)
            logits = outputs.logits
            probabilities = torch.sigmoid(logits)  # Convert logits to probabilities

        probabilities = probabilities.cpu().numpy()
        predictions = (probabilities >=0.25).astype(int)
        all_probabilities.append(probabilities)
        all_predictions.append(predictions)

    # Concatenate predictions from all batches
    probabilities = np.concatenate(all_probabilities, axis=0)
    predictions = (probabilities >=0.25).astype(int)

    return predictions, probabilities


###################################
# 2) Predict on the entire TEST dataset
###################################

# Prepare test texts and labels
test_texts = df["text"].tolist()  # The text column in your test dataset

predictions, probabilities = predict_with_model(model_best, tokenizer, test_texts, device, batch_size=8) # You can adjust batch_size


In [ ]:
predictions

In [ ]:
df["Issuebezogen"] = predictions.tolist()
df = pd.DataFrame(df)

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
df[["author","text","Issuebezogen"]].sample(10)

In [ ]:
df[["D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]] = pd.DataFrame(df["Issuebezogen"].tolist(), index=df.index)

In [ ]:
df[["text","Issuebezogen","D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]]

In [ ]:
df.to_csv("kandis_vercodet.csv")